![image.png](https://i.imgur.com/a3uAqnb.png)

# LLM-as-a-Judge: Position Bias and Agreement with Human Experts

- **Dataset**: [MT-Bench human judgments](https://huggingface.co/datasets/lmsys/mt_bench_human_judgments) from Hugging Face: expert and GPT-4 votes on pairs of chat answers
- **Task**: pairwise judging of chat answers, scored against expert votes
- **Model**: Qwen3-0.6B and Qwen3-1.7B as zero-shot judges; nothing is trained, the judges are measured
- **Runtime**: Colab T4 GPU, about 8 minutes of judging

In this lab, we will:

- build a pairwise judge from a chat model's next-token probabilities
- swap the two answers and measure how often the verdict survives
- score the judges against expert votes, next to GPT-4 and the experts
- cancel position bias by judging both orders, and see where a judge can be trusted

## What is an LLM judge?

Open-ended answers have no answer key, and people grade slowly. An **LLM judge** is a language model prompted to grade instead. In the **pairwise** protocol it reads a question and two answers, Assistant A and Assistant B, and answers A, B, or C for a tie.

A judge is an instrument to test before trusting: its verdict should follow the answers, not their order (preferring one slot is **position bias**), and mostly agree with experts.

With answer $x$ shown first and $y$ second, and $p_A$, $p_B$ the judge's next-token probabilities of the letters A and B, the **margin** splits in two:

$$m(x, y) = \log p_A - \log p_B = \underbrace{\tfrac{1}{2}\big[m(x, y) - m(y, x)\big]}_{d(x,\,y)} + \underbrace{\tfrac{1}{2}\big[m(x, y) + m(y, x)\big]}_{o(x,\,y)}$$

Where:
- $m(x, y) > 0$ means the judge prefers the first answer
- $d$, the **content margin**, flips sign when the answers swap: the preference for $x$ over $y$
- $o$, the **position offset**, stays the same: the preference for the first slot, zero for an unbiased judge

Paper: [Zheng et al., 2023. Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena](https://arxiv.org/abs/2306.05685v4)

## Setup

In Google Colab, select **Runtime > Change runtime type > T4 GPU**, then run the cells from top to bottom.

In [ ]:
import gc
import itertools
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub.utils import logging as hf_logging
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, cohen_kappa_score
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

import torch

hf_logging.set_verbosity_error()   # the dataset and the models are public: hide the notice that no Hugging Face token is set

In [ ]:
SEED = 42
JUDGES = {"Qwen3-0.6B": "Qwen/Qwen3-0.6B", "Qwen3-1.7B": "Qwen/Qwen3-1.7B"}   # short name: Hugging Face id
N_PAIRS = 1_000          # answer pairs to judge, at most; the first turn has 821 with expert votes, so all are judged
BATCH_SIZE = 8           # prompts per forward pass; halve it if the GPU runs out of memory

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
else:
    print("No GPU found, so judging will be slow. In Colab: Runtime > Change runtime type > T4 GPU.")

# The judges run in half precision on the GPU: float16 on a T4, which has no bfloat16 units, and
# bfloat16 on an A100 or newer. On the CPU they run in float32.
MODEL_DTYPE = torch.float32
if DEVICE == "cuda":
    MODEL_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print("judge weights:", MODEL_DTYPE)

## 1. Dataset

- **Source**: [lmsys/mt_bench_human_judgments](https://huggingface.co/datasets/lmsys/mt_bench_human_judgments) on Hugging Face, released with Zheng et al. (2023)
- **Content**: answers by six 2023 models, from GPT-4 to the base model LLaMA-13B, to the 80 two-turn MT-Bench questions in eight categories. `human`: 3,355 votes by 58 expert labellers, mostly graduate students, and by the authors; `gpt4_pair`: GPT-4's verdicts on all 2,400 answer pairs (80 questions, 15 model pairs, 2 turns)
- **Licence**: CC BY 4.0
- **What we use**: first-turn answers and expert votes, as in the paper's agreement table: 1,343 votes on 821 answer pairs

In [ ]:
raw = load_dataset("lmsys/mt_bench_human_judgments")
print(raw)
human = raw["human"].to_pandas()
human[["question_id", "model_a", "model_b", "winner", "judge", "turn"]].head()

Each row is one vote on one turn by an expert (or an author, `author_*`): `model_a`, `model_b` or `tie`. We keep the experts' first-turn votes, record the winner's name, and key each vote by its **answer pair**: the question and both models, alphabetically, whatever order the expert saw.

In [ ]:
# First-turn votes only: the prompt holds one question and two answers (conv[0] and conv[1]);
# judging the second turn would need the whole conversation in the prompt.
votes = human[(human.turn == 1) & human.judge.str.startswith("expert")].copy()
votes["question"] = votes.conversation_a.map(lambda conv: conv[0]["content"])
votes["answer_a"] = votes.conversation_a.map(lambda conv: conv[1]["content"])
votes["answer_b"] = votes.conversation_b.map(lambda conv: conv[1]["content"])
votes["winner_model"] = np.where(votes.winner == "model_a", votes.model_a,
                                 np.where(votes.winner == "model_b", votes.model_b, "tie"))
votes["pair_id"] = [f"{q}:{min(a, b)}:{max(a, b)}" for q, a, b in zip(votes.question_id, votes.model_a, votes.model_b)]
print(f"{len(votes):,} first-turn expert votes by {votes.judge.nunique()} experts on {votes.pair_id.nunique():,} answer pairs")
print("experts voting on the same answer pair:", votes.groupby("pair_id").size().value_counts().sort_index().to_dict())

Then one row per answer pair, its models named `model_1` and `model_2` in random order, so the first slot does not always hold the same model. Votes, and later verdicts, become `"1"`, `"2"` or `"tie"`.

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for pair_id, group in votes.groupby("pair_id", sort=False):
    vote = group.iloc[0]                                  # every vote on a pair carries the same two answers
    texts = {vote.model_a: vote.answer_a, vote.model_b: vote.answer_b}
    model_1, model_2 = rng.permutation(sorted(texts))
    rows.append({"pair_id": pair_id, "question_id": vote.question_id, "question": vote.question,
                 "model_1": model_1, "model_2": model_2, "answer_1": texts[model_1], "answer_2": texts[model_2]})
pairs = pd.DataFrame(rows).set_index("pair_id")
pairs = pairs.sample(n=min(N_PAIRS, len(pairs)), random_state=SEED)
pairs["model_pair"] = [" vs ".join(sorted(m)) for m in zip(pairs.model_1, pairs.model_2)]

votes = votes[votes.pair_id.isin(pairs.index)].reset_index(drop=True)
votes["model_1"] = votes.pair_id.map(pairs.model_1)
votes["model_2"] = votes.pair_id.map(pairs.model_2)
votes["model_pair"] = votes.pair_id.map(pairs.model_pair)
votes["vote"] = np.where(votes.winner_model == "tie", "tie", np.where(votes.winner_model == votes.model_1, "1", "2"))
votes = votes[["pair_id", "judge", "model_1", "model_2", "model_pair", "winner_model", "vote"]]
print(f"{len(pairs):,} answer pairs, {len(votes):,} expert votes")
print("votes:", votes.vote.value_counts().to_dict())

### A look at the data

One answer pair with its votes, then each model's median answer length and **win rate**, the share of its non-tie votes it won.

In [ ]:
example = pairs.iloc[0]
print("QUESTION:", example.question, "\n")
for k in ("1", "2"):
    answer = example[f"answer_{k}"]
    print(f"ANSWER {k} ({example[f'model_{k}']}, {len(answer):,} characters):")
    print(answer[:500] + (" [...]" if len(answer) > 500 else ""), "\n")
print("expert votes:", votes[votes.pair_id == example.name].vote.tolist())

answers = pd.concat([
    pairs[["question_id", f"model_{k}", f"answer_{k}"]].set_axis(["question_id", "model", "answer"], axis=1)
    for k in ("1", "2")
]).drop_duplicates(["question_id", "model"])
answers["characters"] = answers.answer.str.len()
decided = votes[votes.vote != "tie"]
appearances = pd.concat([decided.model_1, decided.model_2]).value_counts()
wins = decided.winner_model.value_counts().reindex(appearances.index, fill_value=0)
model_table = pd.DataFrame({"median answer characters": answers.groupby("model").characters.median(),
                            "expert win rate": wins / appearances})
model_table.sort_values("expert win rate", ascending=False).round(3)

## 2. Reference Points: Chance, Experts and GPT-4

**Agreement** is the share of expert votes a judge matches on the same pair. Zheng et al. (2023) use two setups: **S1** counts every vote, ties included (chance: 1/3); **S2** keeps votes where neither expert nor judge says tie (chance: 1/2). **Cohen's kappa** corrects S1 for chance:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

Where:
- $p_o$ is the observed agreement
- $p_e$ is the agreement expected by chance, each side voting at its own rates of "1", "2" and "tie"; $\kappa = 0$ is chance, $\kappa = 1$ perfect

**Picks longer** is the share of non-tie verdicts, or expert votes, choosing the longer answer.

In [ ]:
def agreement(verdicts, votes, ties=True):
    """Share of expert votes that match the verdict on the same answer pair, and the number of votes
    counted. verdicts: Series of "1", "2" or "tie" indexed by pair_id. ties=False is setup S2: votes
    and verdicts that are ties are left out."""
    v = votes.assign(verdict=votes.pair_id.map(verdicts))
    if not ties:
        v = v[(v.vote != "tie") & (v.verdict != "tie")]
    return (v.vote == v.verdict).mean(), len(v)


def kappa(verdicts, votes):
    """Cohen's kappa between the expert votes and the verdicts, ties included."""
    return cohen_kappa_score(votes.vote, votes.pair_id.map(verdicts), labels=["1", "2", "tie"])


def expert_agreement(votes, ties=True):
    """Agreement of two different experts on the same answer pair, over every such pair of votes."""
    same = total = 0
    for _, group in votes.groupby("pair_id"):
        for v1, v2 in itertools.combinations(group.vote, 2):
            if ties or "tie" not in (v1, v2):
                same += v1 == v2
                total += 1
    return (same / total if total else float("nan")), total


chars_1, chars_2 = pairs.answer_1.str.len(), pairs.answer_2.str.len()
longer_verdict = pd.Series(np.select([chars_1 > chars_2, chars_2 > chars_1], ["1", "2"], "tie"), index=pairs.index)


def picks_longer(verdicts):
    """Share of the non-tie verdicts that pick the longer answer, over pairs whose answers differ in length."""
    v = verdicts[(verdicts != "tie") & (longer_verdict != "tie")]
    return (v == longer_verdict[v.index]).mean()


def report(name, verdicts):
    """One row of the agreement table."""
    s1, _ = agreement(verdicts, votes, ties=True)
    s2, n2 = agreement(verdicts, votes, ties=False)
    return {"judge": name, "S1 agreement": s1, "S2 agreement": s2, "S2 votes": n2, "kappa": kappa(verdicts, votes),
            "says tie": (verdicts == "tie").mean(), "picks longer": picks_longer(verdicts)}

The cell below builds the reference table:

- **two experts** on the same pair: how often people agree (not a ceiling: a judge voting with the majority can beat it)
- **GPT-4**, judged in both orders, with a tie when they disagree (`tie (inconsistent)`); in [the authors' code](https://github.com/lm-sys/FastChat/blob/main/fastchat/llm_judge/common.py#L31), math, reasoning and coding questions also get a reference answer
- a **length heuristic** that picks the longer answer

In [ ]:
gpt4 = raw["gpt4_pair"].to_pandas()
gpt4 = gpt4[gpt4.turn == 1].copy()
gpt4["pair_id"] = [f"{q}:{min(a, b)}:{max(a, b)}" for q, a, b in zip(gpt4.question_id, gpt4.model_a, gpt4.model_b)]
gpt4 = gpt4.set_index("pair_id").reindex(pairs.index)     # the same answer pairs, in the same order
gpt4_winner = np.where(gpt4.winner == "model_a", gpt4.model_a, np.where(gpt4.winner == "model_b", gpt4.model_b, "tie"))
gpt4_verdict = pd.Series(np.where(gpt4_winner == "tie", "tie", np.where(gpt4_winner == pairs.model_1, "1", "2")),
                         index=pairs.index)
print("GPT-4 verdicts:", gpt4_verdict.value_counts().to_dict(),
      "| pairs where its two orders disagreed:", (gpt4.winner == "tie (inconsistent)").sum())

s1, _ = expert_agreement(votes, ties=True)
s2, n2 = expert_agreement(votes, ties=False)
expert_decided = votes[(votes.vote != "tie") & (votes.pair_id.map(longer_verdict) != "tie")]
reference = pd.DataFrame([
    {"judge": "random", "S1 agreement": 1 / 3, "S2 agreement": 1 / 2, "kappa": 0.0, "picks longer": 1 / 2},
    {"judge": "two experts", "S1 agreement": s1, "S2 agreement": s2, "S2 votes": n2,     # S2 votes: pairs of votes
     "picks longer": (expert_decided.vote == expert_decided.pair_id.map(longer_verdict)).mean()},
    report("longer answer wins", longer_verdict),
    report("GPT-4, both orders", gpt4_verdict),
]).set_index("judge")
reference = reference[["S1 agreement", "S2 agreement", "S2 votes", "kappa", "says tie", "picks longer"]]
reference.round(3)

**What to notice**

With all 821 pairs, GPT-4's agreements match the paper's Table 5a (first turn, G4-Pair against Human: 66% in S1, 85% in S2) and exceed the agreement between two experts. The length heuristic is well above chance because Alpaca-13B and LLaMA-13B write far shorter answers than the rest: a judge must beat it, not just chance.

## 3. The Judge

We use the pairwise prompt of Zheng et al. (2023, Figure 5) with one change: their judge explains first, ours gives the verdict at once, so one forward pass suffices. The reply starts with `[[`, so the next token is the verdict letter; `enable_thinking=False` turns off Qwen3's reasoning mode.

In [ ]:
SYSTEM_PROMPT = (
    "Please act as an impartial judge and evaluate the quality of the responses provided by two AI assistants "
    "to the user question displayed below. You should choose the assistant that follows the user's instructions "
    "and answers the user's question better. Your evaluation should consider factors such as the helpfulness, "
    "relevance, accuracy, depth, creativity, and level of detail of their responses. Avoid any position biases "
    "and ensure that the order in which the responses were presented does not influence your decision. Do not "
    "allow the length of the responses to influence your evaluation. Do not favor certain names of the "
    "assistants. Be as objective as possible. Output your final verdict by strictly following this format: "
    '"[[A]]" if assistant A is better, "[[B]]" if assistant B is better, and "[[C]]" for a tie.'
)
USER_TEMPLATE = (
    "[User Question]\n{question}\n\n"
    "[The Start of Assistant A's Answer]\n{answer_a}\n[The End of Assistant A's Answer]\n\n"
    "[The Start of Assistant B's Answer]\n{answer_b}\n[The End of Assistant B's Answer]"
)


def build_prompt(tokenizer, question, first_answer, second_answer):
    """Chat-formatted judge prompt with first_answer as Assistant A, ending in "[[" so that the next
    token is the verdict letter."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_TEMPLATE.format(question=question, answer_a=first_answer,
                                                                 answer_b=second_answer)}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return text + "[["


tokenizer = AutoTokenizer.from_pretrained(next(iter(JUDGES.values())))
example_prompt = build_prompt(tokenizer, example.question, example.answer_1, example.answer_2)
print("[...]" + example_prompt[-600:])
print(f"\n{len(tokenizer(example_prompt).input_ids):,} tokens in this prompt")
print("tokens of '[[A]]':", [tokenizer.decode(t) for t in tokenizer("[[A]]").input_ids])

`verdict_log_probs` keeps the last position's logits only (`logits_to_keep=1`), renormalises A, B and C over the three, and records their share of the full next-token probability, a check that the judge follows the format. Batches group prompts of similar length.

In [ ]:
@torch.no_grad()
def verdict_log_probs(model, tokenizer, prompts):
    """Log-probabilities of the letters A, B and C, renormalised over the three, shape (n, 3), and the
    share of the full next-token probability that falls on them, shape (n,)."""
    letter_ids = tokenizer.convert_tokens_to_ids(["A", "B", "C"])
    lengths = [len(ids) for ids in tokenizer(prompts, add_special_tokens=False).input_ids]
    log_probs = np.zeros((len(prompts), 3), dtype=np.float32)
    letter_mass = np.zeros(len(prompts), dtype=np.float32)
    order = np.argsort(lengths)                    # prompts of similar length share a batch: little padding
    for start in tqdm(range(0, len(prompts), BATCH_SIZE), desc="judging", leave=False):
        idx = order[start:start + BATCH_SIZE]
        batch = tokenizer([prompts[i] for i in idx], return_tensors="pt", padding=True,
                          add_special_tokens=False).to(DEVICE)
        # no key-value cache: nothing is generated, and a long batch's cache would cost gigabytes
        logits = model(**batch, use_cache=False, logits_to_keep=1).logits[:, -1].float()   # (B, vocabulary size)
        log_probs[idx] = torch.log_softmax(logits[:, letter_ids], dim=-1).cpu().numpy()
        letter_mass[idx] = torch.softmax(logits, dim=-1)[:, letter_ids].sum(dim=-1).cpu().numpy()
    return log_probs, letter_mass


def run_judge(model_id):
    """Judges every answer pair in both orders: call 12 shows answer 1 as Assistant A, call 21 shows
    answer 2 as Assistant A. The model lives only inside this function, so its memory is released
    when the function returns."""
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.padding_side = "left"           # so position -1 is every prompt's last real token
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=MODEL_DTYPE).to(DEVICE).eval()
    triples = list(zip(pairs.question, pairs.answer_1, pairs.answer_2))
    calls = {"12": [build_prompt(tokenizer, q, a1, a2) for q, a1, a2 in triples],
             "21": [build_prompt(tokenizer, q, a2, a1) for q, a1, a2 in triples]}
    result = {}
    for call, prompts in calls.items():
        result[call], result[f"mass {call}"] = verdict_log_probs(model, tokenizer, prompts)
    return result

## 4. Judging Every Pair in Both Orders

Call 12 shows answer 1 first, call 21 answer 2. Weights download on first use (about 1.5 GB and 4 GB).

Expected time: about 8 minutes on a T4, downloads included, mostly for Qwen3-1.7B.

In [ ]:
print(f"{len(JUDGES)} judges x 2 orders x {len(pairs):,} answer pairs = {2 * len(JUDGES) * len(pairs):,} prompts, "
      f"{int(np.ceil(len(pairs) / BATCH_SIZE))} batches per judge and order")
judgments = {}
start = time.time()
for name, model_id in JUDGES.items():
    judgments[name] = run_judge(model_id)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    mass = np.concatenate([judgments[name]["mass 12"], judgments[name]["mass 21"]])
    print(f"{name}: mean probability on A, B or C {mass.mean():.3f} (lowest {mass.min():.3f}) | "
          f"{time.time() - start:.0f}s elapsed")

## 5. Position Bias

A judge is **consistent** on a pair when both calls name the same winner, or both say tie. Otherwise it is **biased toward first** if it picks the first-shown answer both times, **toward second** if the second. For GPT-4 we only know which pairs were inconsistent. On very similar GPT-3.5 answers, Zheng et al. (2023, Table 2) found GPT-4 consistent on only 65.0%; our pairs differ more.

In [ ]:
LETTER_TO_VERDICT = {"12": np.array(["1", "2", "tie"]),   # call 12: A is answer 1, B is answer 2, C a tie
                     "21": np.array(["2", "1", "tie"])}   # call 21: A is answer 2, B is answer 1


def call_verdicts(j, call):
    """Verdict of one call, "1", "2" or "tie", indexed by pair_id."""
    return pd.Series(LETTER_TO_VERDICT[call][j[call].argmax(axis=1)], index=pairs.index)


def margin(j, call):
    """m = log p_A - log p_B: > 0 when the judge prefers the answer shown first in that call."""
    return j[call][:, 0] - j[call][:, 1]


rows = []
for name, j in judgments.items():
    letter_12, letter_21 = j["12"].argmax(axis=1), j["21"].argmax(axis=1)
    consistent = (call_verdicts(j, "12") == call_verdicts(j, "21")).mean()
    first = ((letter_12 == 0) & (letter_21 == 0)).mean()
    second = ((letter_12 == 1) & (letter_21 == 1)).mean()
    rows.append({"judge": name, "consistent": consistent, "biased toward first": first, "biased toward second": second,
                 "one call says tie": 1 - consistent - first - second,
                 "mean position offset": ((margin(j, "12") + margin(j, "21")) / 2).mean(),
                 "median |o|": np.median(np.abs(margin(j, "12") + margin(j, "21")) / 2),
                 "median |d|": np.median(np.abs(margin(j, "12") - margin(j, "21")) / 2)})
rows.append({"judge": "GPT-4", "consistent": (gpt4.winner != "tie (inconsistent)").mean()})
pd.DataFrame(rows).set_index("judge").round(3)

Each point is one answer pair, the margin $m_{12}$ of call 12 against $m_{21}$ of call 21, coloured by the experts' majority. An unbiased judge keeps every point on the dashed line $m_{21} = -m_{12}$; an offset $o$ adds $o$ to both margins, leaving the point $2o$ above it (below if $o < 0$), as the red line shows for the mean offset.

In [ ]:
# the experts' majority vote per pair; a split vote counts as a tie
majority = votes.groupby("pair_id").vote.agg(lambda v: v.mode()[0] if v.mode().size == 1 else "tie").reindex(pairs.index)

fig, axes = plt.subplots(1, len(judgments), figsize=(5.5 * len(judgments), 5), squeeze=False)
for ax, (name, j) in zip(axes[0], judgments.items()):
    m12, m21 = margin(j, "12"), margin(j, "21")
    offset = ((m12 + m21) / 2).mean()
    lim = 1.05 * max(np.abs(m12).max(), np.abs(m21).max(), 2 * abs(offset), 1e-3)
    line = np.array([-lim, lim])
    for label, color in [("tie", "lightgrey"), ("1", "tab:blue"), ("2", "tab:orange")]:
        mask = (majority == label).to_numpy()
        ax.scatter(m12[mask], m21[mask], s=8, alpha=0.6, color=color, label=f"experts: {label}")
    ax.plot(line, -line, "k--", lw=1, label="no position bias: $m_{21} = -m_{12}$")
    ax.plot(line, -line + 2 * offset, "r-", lw=1, label="shifted by twice the mean offset")
    ax.axhline(0, color="grey", lw=0.5)
    ax.axvline(0, color="grey", lw=0.5)
    ax.set(xlim=(-lim, lim), ylim=(-lim, lim), xlabel="$m_{12}$: margin with answer 1 shown first",
           ylabel="$m_{21}$: margin with answer 2 shown first", title=f"{name}: mean position offset {offset:+.2f}")
    ax.legend(loc="best", fontsize=7)
plt.tight_layout()
plt.show()

**What to notice**

Qwen3-0.6B picks the second answer in most pairs, Qwen3-1.7B leans to the first, and GPT-4 is far more consistent: the direction belongs to the judge. Leaving ties aside, a pair is consistent exactly when $|d| > |o|$, since $m_{12} = d + o$ and $m_{21} = -d + o$ then differ in sign; Qwen3-1.7B's offsets are no smaller, but its content margins are far wider. Under the offset the margins follow the answers: pairs the experts gave to answer 1 sit lower right.

## 6. Agreement with Human Experts

Three ways to turn the two calls into one verdict:

- **one call**: call 12 alone, cheapest and most exposed to position bias
- **swap and tie**: a winner when both calls agree, else a tie, as for GPT-4
- **averaged margin**: the sign of $d = \tfrac{1}{2}(m_{12} - m_{21})$, free of each pair's offset $o$; never a tie

In [ ]:
def verdicts_one_call(j):
    return call_verdicts(j, "12")


def verdicts_swap_and_tie(j):
    v12, v21 = call_verdicts(j, "12"), call_verdicts(j, "21")
    return v12.where(v12 == v21, "tie")


def verdicts_averaged(j):
    d = (margin(j, "12") - margin(j, "21")) / 2      # > 0: answer 1 preferred over both orders together
    return pd.Series(np.where(d > 0, "1", "2"), index=pairs.index)


RULES = {"one call": verdicts_one_call, "swap and tie": verdicts_swap_and_tie, "averaged margin": verdicts_averaged}
rows = [report(f"{name}, {rule}", make(j)) for name, j in judgments.items() for rule, make in RULES.items()]
results = pd.concat([reference, pd.DataFrame(rows).set_index("judge")])
results.round(3)

A judge is a classifier of answer pairs: below, the larger judge's averaged verdicts against every expert vote, with precision and recall per class.

In [ ]:
largest = list(judgments)[-1]                     # the last judge in JUDGES, the largest model
y_true, y_pred = votes.vote, votes.pair_id.map(verdicts_averaged(judgments[largest]))
print(classification_report(y_true, y_pred, labels=["1", "2", "tie"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, labels=["1", "2", "tie"], cmap="Blues")
plt.title(f"{largest}, averaged margin")
plt.xlabel("judge verdict")
plt.ylabel("expert vote")
plt.show()

**What to notice**

One call leaves Qwen3-0.6B near chance. Swap and tie lifts its S2 to about GPT-4's level only by calling most pairs a tie: S2 leaves those votes out, kappa counts them. The averaged margin gives each judge its best kappa, above the length heuristic and below GPT-4. With both orders, every judge, GPT-4 included, picks the longer answer more often than the experts.

## 7. Where Can a Judge Be Trusted?

Zheng et al. (2023, Figure 2) found GPT-4's agreement with humans rising from about 70% on close model pairs to nearly 100% on distant ones. Per model pair, we plot S2 agreement against the **win-rate gap** $|2w - 1|$, where $w$ is one model's share of the pair's non-tie expert votes.

In [ ]:
rows = []
for model_pair, group in votes.groupby("model_pair"):
    decided = group[group.vote != "tie"]
    w = (decided.winner_model == model_pair.split(" vs ")[0]).mean()
    row = {"model pair": model_pair, "win-rate gap": abs(2 * w - 1), "non-tie votes": len(decided)}
    for name, j in judgments.items():
        row[name] = agreement(verdicts_averaged(j), group, ties=False)[0]
    row["GPT-4"] = agreement(gpt4_verdict, group, ties=False)[0]
    row["longer answer wins"] = agreement(longer_verdict, group, ties=False)[0]
    rows.append(row)
by_model_pair = pd.DataFrame(rows).set_index("model pair").sort_values("win-rate gap")

fig, ax = plt.subplots(figsize=(8, 5))
for column in [*judgments, "GPT-4", "longer answer wins"]:
    ax.plot(by_model_pair["win-rate gap"], by_model_pair[column], marker="o", lw=1, label=column)
ax.axhline(0.5, color="grey", ls="--", lw=1, label="chance (S2)")
ax.set(xlabel="win-rate gap |2w - 1| of the model pair", ylabel="S2 agreement with experts",
       title="Agreement per model pair (our judges: averaged margin)")
ax.legend()
plt.show()
by_model_pair.round(3)

**What to notice**

Agreement rises with the win-rate gap for every judge, the length heuristic included, but on close pairs the small judges fall towards chance while GPT-4 keeps a lead. Close pairs are what a leaderboard must settle, so a judge is most trustworthy where it is needed least.

## 8. Save and Reload

Judging is the expensive part: we save the letters' log-probabilities, reload them, and recompute one result.

In [ ]:
saved = pairs[["question_id", "model_1", "model_2"]].copy()
for name, j in judgments.items():
    for call in ("12", "21"):
        for k, letter in enumerate("ABC"):
            saved[f"{name} call {call} log p({letter})"] = j[call][:, k]
saved.to_csv("judge_log_probs.csv")

reloaded = pd.read_csv("judge_log_probs.csv", index_col="pair_id").reindex(pairs.index)
for name, j in judgments.items():
    # read back as float32, the precision the judge produced, so every margin matches to the last bit
    j_reloaded = {call: reloaded[[f"{name} call {call} log p({letter})" for letter in "ABC"]].to_numpy(dtype=np.float32)
                  for call in ("12", "21")}
    before = agreement(verdicts_averaged(j), votes, ties=False)[0]
    after = agreement(verdicts_averaged(j_reloaded), votes, ties=False)[0]
    print(f"{name}: S2 agreement of the averaged margin {before:.3f} before saving, {after:.3f} from the file")

## Summary

- Test an LLM judge for position bias and against human labels before trusting it.
- Next-token probabilities give a verdict in one forward pass, and a margin with content and position parts.
- Small judges can be strongly position-biased, either way; the content margin of both orders removes the offset.
- S2 drops ties, so read it with its vote count; kappa corrects for chance.
- Judges agree with experts most where they are needed least: on model pairs far apart in quality.

## Exercises

1. **A larger judge.** In a new cell, set `BATCH_SIZE = 4` and run `judgments["Qwen3-4B-Instruct-2507"] = run_judge("Qwen/Qwen3-4B-Instruct-2507")` (a later Qwen3-4B update, 8 GB in float16), then rerun from section 5. Is it more consistent because its offset shrinks or its content margins grow?
2. **A tie band.** Let the averaged margin say tie when $|d| < \tau$. Tune $\tau$ for kappa on the odd-numbered questions and report S1, S2 and kappa on the even ones. Why not tune on the scored pairs?
3. **Explanation first.** Restore the original prompt's explanation step (Zheng et al., 2023, Figure 5), drop the `[[` prefix, and generate verdicts for 50 pairs in both orders. Are they more consistent?
4. **Padding.** Judge 20 answers against copies that repeat their first paragraph at the end, in both orders. Which judges prefer the copy?

## Further Reading

- [Zheng et al., 2023. Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena](https://arxiv.org/abs/2306.05685v4)
- [Wang et al., 2024. Large Language Models are not Fair Evaluators](https://arxiv.org/abs/2305.17926)
- [Shi et al., 2025. Judging the Judges: A Systematic Study of Position Bias in LLM-as-a-Judge](https://arxiv.org/abs/2406.07791)

### Contributed by: Ali Habibullah